# 07 — Validação e teste com dias de 2026 (protocolo congelado)
Roda **do começo ao fim**: triagem dos dias → download → features → escolha do limiar na validação → teste uma única vez.
Basta executar as células **em ordem** (Ambiente de execução → Executar tudo, ou célula a célula).

## Protocolo (fixo, escrito antes de ver qualquer resultado)
| Item | Regra |
|---|---|
| Dias candidatos | dia 15 de cada mês, janeiro a outubro de 2026 (todos posteriores ao treino de março/2025) |
| Dia avaliável | pelo menos **2 instâncias** do Tornado com ≥ 10 transações no dia (diretas + via Router, estimado na triagem) |
| Quantos dias | **modo escolhido:** 1 dia novo de teste (`DIA_TESTE_FIXO` = 15/01/2026). Modo antigo (`DIA_TESTE_FIXO = None`): no máximo `MAX_DIAS` dias de 2026 |
| Papéis | **validação = 29/10/2020** (dia do artigo, já processado: só escolhe o limiar); **teste = 15/01/2026** (dia novo, rodado uma vez) |
| Modelo | o `modelo_rf.joblib` do notebook 03, **sem retreinar** |
| Regra do limiar | grade 0,50 a 0,95 (passo 0,05); vale o de **maior F1** nos dias de validação; empate → o **maior** limiar |
| Teste | o limiar é congelado e aplicado **uma vez** aos dias de teste |
| Router do Tornado | fica fora do ranking e da contagem de falsos positivos (é infraestrutura do próprio mixer); é listado à parte |
| Falsos positivos | = endereços que **não são** instâncias do Tornado; **não verificados** (a precisão é uma estimativa pessimista) |

⚠️ **Não altere** os parâmetros depois de ver os resultados. Se precisar mudar algo, registre a mudança e marque o resultado como exploratório.

⏱️ Os downloads são lentos (~40 min por dia, ~250–300 MB). Mantenha a aba aberta. Se a sessão cair, execute de novo: tudo continua de onde parou.

## Configuração

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, numpy as np, json, time, requests, gc, subprocess, warnings, datetime
warnings.filterwarnings("ignore", category=FutureWarning)

BASE = Path('/content/drive/MyDrive/mixers-ethereum')
RAW, INTERIM, PROC, RES = [BASE/p for p in ['data/raw','data/interim','data/processed','results']]
for p in [RAW, INTERIM, PROC, RES, BASE/'docs']:
    p.mkdir(parents=True, exist_ok=True)

API = "https://api.blockchair.com/ethereum"
SEED = 42
TORNADO = {
    "0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc": "0.1 ETH",
    "0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936": "1 ETH",
    "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf": "10 ETH",
}
FEATURES = ["value_entropy", "fee_entropy", "internal_value_entropy", "call_count_mean"]
MIN_TX = 10
DECIMAIS = {"value_eth": 4, "internal_value_eth": 4, "fee_eth": 6}   # mesmos do notebook 03

# ===== PARÂMETROS DO PROTOCOLO (não mudar depois de ver resultados) =====
DIAS_CANDIDATOS = [f"2026-{m:02d}-15" for m in range(1, 11)]
MAX_DIAS = 4          # mais dias = resultado mais firme, porém mais download (~40 min cada)
MIN_INST = 2          # instâncias avaliáveis mínimas para o dia entrar
GRADE = [round(x, 2) for x in np.arange(0.50, 0.951, 0.05)]
FRACAO_VALIDACAO = 0.4   # só no modo antigo (vários dias de 2026)
# ===== MODO ESCOLHIDO: 1 dia novo de teste; a validação usa o dia de 2020 já processado =====
DIA_TESTE_FIXO = "2026-01-15"   # None = modo antigo
VALIDACAO_2020 = "2020-10-29"   # features em data/processed/features_teste_20201029.parquet (notebook 04); None = sem validação externa

dec = BASE/"docs/decisoes.md"
if not dec.exists():
    dec.write_text("# Decisões de reprodução\n\n| # | Tema | Decisão | Justificativa | Data |\n|---|---|---|---|---|\n")
def registrar(linha):
    if linha not in dec.read_text():
        with open(dec, "a") as fh:
            fh.write(linha + "\n")
print("OK:", BASE, "| candidatos:", DIAS_CANDIDATOS)

Mounted at /content/drive
OK: /content/drive/MyDrive/mixers-ethereum | candidatos: ['2026-01-15', '2026-02-15', '2026-03-15', '2026-04-15', '2026-05-15', '2026-06-15', '2026-07-15', '2026-08-15', '2026-09-15', '2026-10-15']


## Funções (features, seleção de dias, métricas)

In [2]:
from sklearn.metrics import average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# ---------- features (idênticas às do notebook 03) ----------
def preparar(df):
    df = df.copy()
    for c in ["value", "internal_value", "fee"]:
        if c + "_eth" not in df.columns:
            df[c + "_eth"] = pd.to_numeric(df[c], errors="coerce") / 1e18
    df["time"] = pd.to_datetime(df["time"])
    df["dia"] = df["time"].dt.date
    df["recipient"] = df["recipient"].astype(str).str.lower()
    df["call_count"] = pd.to_numeric(df["call_count"], errors="coerce")
    return df[["hash", "time", "dia", "sender", "recipient", "call_count",
               "value_eth", "internal_value_eth", "fee_eth"]]

def entropia(df, chaves, col, decimais):
    v = df[col].round(decimais)
    cont = df.assign(_v=v).groupby(chaves + ["_v"], observed=True, dropna=False).size()
    p = cont / cont.groupby(level=chaves).transform("sum")
    return (-(p * np.log2(p))).groupby(level=chaves).sum()

def calcular_features(df, chaves=("recipient", "dia")):
    chaves = list(chaves)
    g = df.groupby(chaves, observed=True)
    out = pd.DataFrame({
        "n_tx": g.size(),
        "value_entropy": entropia(df, chaves, "value_eth", DECIMAIS["value_eth"]),
        "fee_entropy": entropia(df, chaves, "fee_eth", DECIMAIS["fee_eth"]),
        "internal_value_entropy": entropia(df, chaves, "internal_value_eth", DECIMAIS["internal_value_eth"]),
        "call_count_mean": g["call_count"].mean(),
    })
    return out.reset_index()

def norm_hash(s):
    return s.astype(str).str.lower().str.replace(r"^0x", "", regex=True)

# ---------- seleção de dias e papéis ----------
def selecionar_dias(tri, max_dias=MAX_DIAS, min_inst=MIN_INST):
    ok = tri[tri.avaliavel_ampliado].groupby("dia").size()
    dias = sorted(ok[ok >= min_inst].index)
    if len(dias) > max_dias:
        idx = np.unique(np.round(np.linspace(0, len(dias) - 1, max_dias)).astype(int))
        dias = [dias[i] for i in idx]
    return dias

def dividir_papeis(dias, fracao=FRACAO_VALIDACAO):
    n = len(dias)
    nv = max(1, int(round(fracao * n)))
    if n - nv < 1:
        nv = n - 1
    return {d: ("validacao" if i < nv else "teste") for i, d in enumerate(dias)}

# ---------- métricas ----------
def contagens(el, lim):
    s = el[el.prob > lim]
    tp = int(s.positivo.sum()); fp = len(s) - tp
    fn = int(el.positivo.sum()) - tp
    return tp, fp, fn

def f1_de(tp, fp, fn):
    d = 2 * tp + fp + fn
    return 2 * tp / d if d > 0 else 0.0

def resumo_limiar(lista_el, lim):
    tp = fp = fn = 0
    for el in lista_el:
        a, b, c = contagens(el, lim); tp += a; fp += b; fn += c
    sin = tp + fp
    return {"limiar": lim, "sinalizados": sin, "tornado": tp, "falsos_positivos": fp, "fn": fn,
            "precisao": round(tp / sin, 3) if sin else np.nan,
            "recall": round(tp / (tp + fn), 3) if (tp + fn) else np.nan,
            "f1": round(f1_de(tp, fp, fn), 3)}

def escolher_limiar(lista_el, grade=GRADE):
    tab = pd.DataFrame([resumo_limiar(lista_el, lim) for lim in grade])
    melhor = tab.f1.max()
    lim = float(tab[np.isclose(tab.f1, melhor)].limiar.max())     # empate -> maior limiar
    return lim, tab

def metricas_ranking(el):
    el = el.sort_values("prob", ascending=False).reset_index(drop=True)
    npos = int(el.positivo.sum())
    ranks = [int((el.prob >= p).sum()) for p in el.loc[el.positivo, "prob"]]   # rank pessimista (empates contam contra)
    return {"janelas": len(el), "positivos_avaliaveis": npos,
            "ranks_tornado": sorted(ranks),
            "precisao_em_k": round(float(el.positivo.iloc[:max(npos, 1)].mean()), 3) if npos else np.nan,
            "precisao_em_10": round(float(el.positivo.iloc[:10].mean()), 3),
            "ap": round(float(average_precision_score(el.positivo, el.prob)), 3) if npos else np.nan}

def novo_modelo(seed=SEED):
    return RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=seed)

def reamostrar(X, y, seed=SEED):
    d = X.assign(_y=y.values); maj, mino = d[d._y == 0], d[d._y == 1]
    if len(mino) < len(maj):
        mino = resample(mino, replace=True, n_samples=len(maj), random_state=seed)
    d = pd.concat([maj, mino]); return d[FEATURES], d["_y"]
print("funções carregadas")

funções carregadas


---
## Etapa 1 — Triagem dos dias pela API (sem baixar nada, sem rodar o modelo)
Conta, para cada dia candidato e instância, as transações diretas e as chamadas internas. A cota gratuita da API é limitada: **se parar por cota, execute de novo mais tarde (ou amanhã)**; o que já foi contado fica guardado.

**Teste de sanidade:** em 29/10/2020 o notebook 01 contou 73, 55 e 111 transações diretas. Se não bater, a célula **para** e nada abaixo deve ser usado.

In [3]:
def contar(tabela, endereco, dia):
    q = f"recipient({endereco}),time({dia}..{dia})"
    r = requests.get(f"{API}/{tabela}", params={"q": q, "limit": 1}, timeout=60).json()
    ctx = r.get("context", {})
    if ctx.get("code") != 200:
        raise RuntimeError(f"API parou ({tabela}, {dia}, {endereco[:8]}): código {ctx.get('code')} - {ctx.get('error')}. "
                           "Execute de novo mais tarde; o progresso fica salvo.")
    time.sleep(3)
    return int(ctx["total_rows"])

esperado = {"0.1 ETH": 73, "1 ETH": 55, "10 ETH": 111}
cache_san = RES/"sanidade_api_2020.json"
if cache_san.exists():
    print("sanidade já verificada:", cache_san.read_text())
else:
    obtido = {nome: contar("transactions", end, "2020-10-29") for end, nome in TORNADO.items()}
    print("API:", obtido, "| esperado (notebook 01):", esperado)
    assert obtido == esperado, "Contagem da API difere do notebook 01: NÃO use os resultados; avise para corrigir o filtro de data."
    cache_san.write_text(json.dumps(obtido))
    print("sanidade OK")

sanidade já verificada: {"0.1 ETH": 73, "1 ETH": 55, "10 ETH": 111}


In [4]:
arq_tri = RES/"triagem_dias_2026.csv"
tri = pd.read_csv(arq_tri) if arq_tri.exists() else pd.DataFrame(
    columns=["dia","instancia","n_direto","n_calls","n_via_estimado","n_total_estimado","avaliavel_conservador","avaliavel_ampliado"])
feitos = set(zip(tri.dia, tri.instancia))
for dia in DIAS_CANDIDATOS:
    for end, nome in TORNADO.items():
        if (dia, nome) in feitos:
            continue
        nd = contar("transactions", end, dia)
        nc = contar("calls", end, dia)
        via = max(nc - nd, 0)
        nova = {"dia": dia, "instancia": nome, "n_direto": nd, "n_calls": nc, "n_via_estimado": via,
                "n_total_estimado": nd + via, "avaliavel_conservador": nd >= MIN_TX,
                "avaliavel_ampliado": (nd + via) >= MIN_TX}
        tri = pd.concat([tri, pd.DataFrame([nova])], ignore_index=True)
        tri.to_csv(arq_tri, index=False)
    print("ok", dia)
tri["avaliavel_ampliado"] = tri["avaliavel_ampliado"].astype(bool)
resumo_tri = (tri.groupby("dia").agg(avaliaveis_conservador=("avaliavel_conservador", "sum"),
                                     avaliaveis_ampliado=("avaliavel_ampliado", "sum"),
                                     tx_total_estimado=("n_total_estimado", "sum")).reset_index())
resumo_tri

ok 2026-01-15
ok 2026-02-15
ok 2026-03-15
ok 2026-04-15
ok 2026-05-15
ok 2026-06-15
ok 2026-07-15
ok 2026-08-15
ok 2026-09-15
ok 2026-10-15


,dia,avaliaveis_conservador,avaliaveis_ampliado,tx_total_estimado
0,2026-01-15,0,3,289
1,2026-02-15,0,3,147
2,2026-03-15,0,3,141
3,2026-04-15,0,3,247
4,2026-05-15,0,3,217
5,2026-06-15,0,3,145
6,2026-07-15,0,3,745
7,2026-08-15,0,3,640
8,2026-09-15,0,3,1039
9,2026-10-15,3,3,482


## Etapa 2 — Escolha dos dias e papéis (congelada)
Aplica a regra do protocolo e grava `results/protocolo_2026.json`. **Se o arquivo já existir, ele é reaproveitado** (os dias não mudam se você rodar de novo).

In [6]:
arq_prot = RES/("protocolo_2026_teste_unico.json" if DIA_TESTE_FIXO else "protocolo_2026.json")
if arq_prot.exists():
    prot = json.loads(arq_prot.read_text())
    print("Protocolo já congelado em", prot["congelado_em"])
else:
    if DIA_TESTE_FIXO:
        linhas_tri = tri[tri.dia == DIA_TESTE_FIXO]
        n_aval = int(linhas_tri.avaliavel_ampliado.sum()) if len(linhas_tri) else 0
        print(f"{DIA_TESTE_FIXO}: instâncias avaliáveis pela triagem = {n_aval}")
        if n_aval < 1:
            raise RuntimeError("O dia de teste escolhido não tem instância avaliável (ou não foi triado). Veja a tabela da Etapa 1.")
        dias = [DIA_TESTE_FIXO]; papeis = {DIA_TESTE_FIXO: "teste"}
    else:
        dias = selecionar_dias(tri)
        if len(dias) < 2:
            raise RuntimeError("Menos de 2 dias avaliáveis em 2026 pela regra do protocolo. Isso é um achado: "
                               "veja a tabela de triagem e discuta com o grupo (mais dias, outra data de corte).")
        papeis = dividir_papeis(dias)
    prot = {"congelado_em": datetime.datetime.now().isoformat(timespec="seconds"),
            "dias": dias, "papeis": papeis, "validacao_externa": VALIDACAO_2020, "min_tx": MIN_TX, "grade": GRADE,
            "regra_limiar": "maior F1 na validacao; empate -> maior limiar"}
    arq_prot.write_text(json.dumps(prot, indent=1))
    registrar(f"| 11 | Dias de avaliação | Validação: {[d for d,p in papeis.items() if p=='validacao'] + ([VALIDACAO_2020] if VALIDACAO_2020 else [])}; teste: {[d for d,p in papeis.items() if p=='teste']} | Download limitado pelo tamanho (>400 MB/dia): 1 dia novo de teste; limiar escolhido no dia de 2020 (antes de ver o teste) | {datetime.date.today()} |")
    registrar("| 12 | Regra do limiar | Máximo F1 na validação (grade 0,50–0,95, passo 0,05), empate → maior limiar; congelado antes do teste | Evita escolher o limiar olhando o teste | " + str(datetime.date.today()) + " |")
DIAS = prot["dias"]; PAPEIS = prot["papeis"]
print(pd.Series(PAPEIS, name="papel").to_frame())

Protocolo já congelado em 2026-11-14T21:47:26
            papel
2026-01-15  teste


## Etapa 3 — Download dos dumps dos dias escolhidos
~250–300 MB por dia, ~40 min cada pelo Colab. Usa `wget -c` (retoma se cair). Cada arquivo é verificado com `gzip -t` e o SHA-256 vai para `docs/coleta.md`.

In [10]:
def nome_dump(dia): return f"blockchair_ethereum_transactions_{dia.replace('-','')}.tsv.gz"

for dia in DIAS:
    f = RAW/nome_dump(dia)
    url = f"https://gz.blockchair.com/ethereum/transactions/{nome_dump(dia)}"
    if f.exists() and subprocess.run(["gzip", "-t", str(f)]).returncode == 0:
        print("já baixado e íntegro:", f.name); continue
    print("baixando", f.name, datetime.datetime.now().strftime("%H:%M"))
    p = subprocess.Popen(["wget", "-c", "--tries=0", "--retry-connrefused", "-q", "-P", str(RAW), url])
    t0, ultimo = time.time(), f.stat().st_size if f.exists() else 0
    while p.poll() is None:
        time.sleep(60)
        atual = f.stat().st_size if f.exists() else 0
        print(f"  {atual/1e6:.0f} MB | {(atual-ultimo)/60/1e3:.0f} KB/s | {(time.time()-t0)/60:.0f} min")
        ultimo = atual
    if not (f.exists() and subprocess.run(["gzip", "-t", str(f)]).returncode == 0):
        raise RuntimeError(f"Download incompleto: {f.name}. Execute a célula de novo (retoma de onde parou).")
    sha = subprocess.run(["sha256sum", str(f)], capture_output=True, text=True).stdout.strip()
    with open(BASE/"docs/coleta.md", "a") as fh:
        fh.write(f"\n## {f.name}\n- URL: {url}\n- Baixado em: {datetime.date.today()}, pelo Colab\n- Tamanho: {f.stat().st_size:,} bytes\n- SHA-256: {sha}\n")
    print("ok:", f.name, f"{f.stat().st_size/1e6:.0f} MB")

baixando blockchair_ethereum_transactions_20260115.tsv.gz 22:08
  550 MB | 102 KB/s | 1 min
  556 MB | 102 KB/s | 2 min
  562 MB | 102 KB/s | 3 min
  568 MB | 103 KB/s | 4 min
  574 MB | 102 KB/s | 5 min
  580 MB | 103 KB/s | 6 min
  587 MB | 103 KB/s | 7 min
  593 MB | 102 KB/s | 8 min
  599 MB | 103 KB/s | 9 min
  605 MB | 102 KB/s | 10 min
  611 MB | 103 KB/s | 11 min
  617 MB | 102 KB/s | 12 min
  623 MB | 102 KB/s | 13 min
  630 MB | 103 KB/s | 14 min
  636 MB | 102 KB/s | 15 min
  642 MB | 103 KB/s | 16 min
  648 MB | 102 KB/s | 17 min
  654 MB | 102 KB/s | 18 min
  660 MB | 103 KB/s | 19 min
  666 MB | 103 KB/s | 20 min
  672 MB | 102 KB/s | 21 min
  679 MB | 103 KB/s | 22 min
  685 MB | 102 KB/s | 23 min
  691 MB | 103 KB/s | 24 min
  697 MB | 102 KB/s | 25 min
  703 MB | 102 KB/s | 26 min
  709 MB | 103 KB/s | 27 min
  715 MB | 102 KB/s | 28 min
  722 MB | 103 KB/s | 29 min
  728 MB | 102 KB/s | 30 min
  734 MB | 102 KB/s | 31 min
  740 MB | 103 KB/s | 32 min
  746 MB | 102 KB

In [11]:
import requests
url = 'https://gz.blockchair.com/ethereum/transactions/blockchair_ethereum_transactions_20260115.tsv.gz'
response = requests.head(url)
tamanho_bytes = int(response.headers.get('content-length', 0))
tamanho_mb = tamanho_bytes / 1e6
print(f'Tamanho total do arquivo: {tamanho_bytes:,} bytes ({tamanho_mb:.2f} MB)')

Tamanho total do arquivo: 752,673,845 bytes (752.67 MB)


In [12]:
import os
import requests

caminho_arquivo = RAW / nome_dump(DIAS[0])
tamanho_atual_bytes = os.path.getsize(caminho_arquivo) if os.path.exists(caminho_arquivo) else 0
tamanho_atual_mb = tamanho_atual_bytes / 1e6

url = f'https://gz.blockchair.com/ethereum/transactions/{nome_dump(DIAS[0])}'
response = requests.head(url)
tamanho_total_bytes = int(response.headers.get('content-length', 0))
tamanho_total_mb = tamanho_total_bytes / 1e6

print(f'Progresso do Download para {nome_dump(DIAS[0])}:')
print(f'- Já baixado (em disco): {tamanho_atual_bytes:,} bytes ({tamanho_atual_mb:.2f} MB)')
print(f'- Tamanho total esperado: {tamanho_total_bytes:,} bytes ({tamanho_total_mb:.2f} MB)')
if tamanho_total_bytes > 0:
    print(f'- Porcentagem concluída: {(tamanho_atual_bytes / tamanho_total_bytes) * 100:.2f}%')


Progresso do Download para blockchair_ethereum_transactions_20260115.tsv.gz:
- Já baixado (em disco): 752,673,845 bytes (752.67 MB)
- Tamanho total esperado: 752,673,845 bytes (752.67 MB)
- Porcentagem concluída: 100.00%


## Etapa 4 — Chamadas via Router (API, só dos dias escolhidos)
No treino, as transações que chegam ao contrato **via Router** foram atribuídas ao contrato. Aqui se faz o mesmo: a API lista as chamadas recebidas por cada instância no dia; as transações que não são diretas são buscadas **dentro do próprio dump**.
Se a cota da API acabar, execute de novo mais tarde: continua do ponto salvo.

In [13]:
def coletar_calls(end, dia, pausa=3):
    nome = TORNADO[end].replace(" ", "")
    saida = RAW/f"calls_{nome}_{dia}.csv"; feito = RAW/f"calls_{nome}_{dia}.done"; estado = RAW/f"calls_{nome}_{dia}.estado.json"
    if feito.exists(): return True
    offset = json.loads(estado.read_text())["offset"] if estado.exists() else 0
    q = f"recipient({end}),time({dia}..{dia})"
    while True:
        r = requests.get(f"{API}/calls", params={"q": q, "limit": 100, "offset": offset}, timeout=60).json()
        ctx = r.get("context", {})
        if ctx.get("code") != 200:
            print(f"Parou em {nome} {dia}, offset {offset}: código {ctx.get('code')} - {ctx.get('error')}"); return False
        dados = r["data"]
        if dados:
            pd.DataFrame(dados).to_csv(saida, mode="a", header=not saida.exists(), index=False)
        offset += len(dados)
        estado.write_text(json.dumps({"offset": offset}))
        print(f"{nome} {dia}: {offset}/{ctx.get('total_rows')}")
        if len(dados) < 100:
            feito.write_text("ok"); return True
        time.sleep(pausa)

todos_ok = True
for dia in DIAS:
    for end, nome in TORNADO.items():
        linha = tri[(tri.dia == dia) & (tri.instancia == nome)]
        if len(linha) and int(linha.n_via_estimado.iloc[0]) == 0 and int(linha.n_direto.iloc[0]) == 0 and int(linha.n_calls.iloc[0]) == 0:
            (RAW/f"calls_{nome.replace(' ','')}_{dia}.done").write_text("ok")   # nada a coletar
            continue
        todos_ok &= coletar_calls(end, dia)
if not todos_ok:
    raise RuntimeError("Coleta de chamadas incompleta (cota da API). Execute esta célula de novo mais tarde.")
print("chamadas coletadas para todos os dias")

0.1ETH 2026-01-15: 74/74
1ETH 2026-01-15: 100/171
1ETH 2026-01-15: 171/171
10ETH 2026-01-15: 44/44
chamadas coletadas para todos os dias


## Etapa 5 — Dump → Parquet → features por janela
Lê o dump em pedaços (economiza RAM), monta as janelas (endereço destinatário, dia UTC) e atribui às instâncias as transações via Router.
O endereço do Router é descoberto nos dados (destinatário mais comum das transações via Router) e fica rotulado como `router`.

In [14]:
def dump_para_parquet(dia):
    f = RAW/nome_dump(dia); saida = INTERIM/f"dia_{dia.replace('-','')}.parquet"
    if saida.exists(): return saida
    COLS = ["hash","time","failed","sender","recipient","call_count","value","internal_value","fee"]
    DT = {"hash": str, "sender": str, "recipient": str, "value": str, "internal_value": str, "fee": str}
    partes = []
    for ch in pd.read_csv(f, sep="\t", usecols=COLS, dtype=DT, chunksize=250_000):
        for c in ["value", "internal_value", "fee"]:
            ch[c + "_eth"] = pd.to_numeric(ch[c], errors="coerce") / 1e18
        partes.append(ch.drop(columns=["value", "internal_value", "fee"]))
        print(f"  {sum(len(p) for p in partes):,} linhas", end="\r")
    df = pd.concat(partes, ignore_index=True); del partes; gc.collect()
    df.to_parquet(saida, index=False)
    return saida

def features_do_dia(dia):
    out = PROC/f"features_{dia}.parquet"
    if out.exists():
        return pd.read_parquet(out)
    df = preparar(pd.read_parquet(dump_para_parquet(dia)))
    df["hash"] = norm_hash(df["hash"])
    extras, routers, log = [], set(), []
    for end, nome in TORNADO.items():
        arq = RAW/f"calls_{nome.replace(' ','')}_{dia}.csv"
        hs = set(norm_hash(pd.read_csv(arq, usecols=["transaction_hash"], dtype=str)["transaction_hash"])) if arq.exists() else set()
        diretos = set(df.loc[df.recipient == end, "hash"])
        via = hs - diretos
        r = df[df.hash.isin(via)].copy()
        if len(r):
            routers |= set(r.recipient.value_counts().head(1).index) - {end}
            r["recipient"] = end
            extras.append(r)
        log.append({"dia": dia, "instancia": nome, "n_direto": len(diretos), "n_via_encontrado": len(r),
                    "n_via_ausente_no_dump": len(via) - len(r)})
    feat = calcular_features(pd.concat([df] + extras, ignore_index=True))
    feat["rotulo"] = feat.recipient.map(TORNADO)
    feat.loc[feat.recipient.isin(routers), "rotulo"] = "router"
    feat.to_parquet(out, index=False)
    pd.DataFrame(log).to_csv(RES/f"atribuicao_router_{dia}.csv", index=False)
    del df, extras; gc.collect()
    return feat

for dia in DIAS:
    ft = features_do_dia(dia)
    print(dia, "| janelas:", f"{len(ft):,}", "| com >= MIN_TX:", f"{(ft.n_tx >= MIN_TX).sum():,}",
          "| Tornado/Router:", ft[ft.rotulo.notna()][["rotulo", "n_tx"]].values.tolist())

2026-01-15 | janelas: 302,716 | com >= MIN_TX: 20,215 | Tornado/Router: [['0.1 ETH', 74], ['1 ETH', 171], ['10 ETH', 43], ['router', 305]]


---
## Etapa 6 — Validação: escolher o limiar
Usa o modelo do notebook 03 (sem retreinar). O limiar é escolhido **só** nos dias de validação (no modo escolhido, o dia de 29/10/2020) pela regra do protocolo e gravado em `results/limiar_congelado_2026.json`.

In [15]:
import joblib
try:
    modelo = joblib.load(PROC/"modelo_rf.joblib")
    print("modelo carregado de modelo_rf.joblib")
except Exception as e:
    print("Não foi possível carregar o modelo salvo (", type(e).__name__, "). Refazendo o treino com os mesmos dados e semente do notebook 03.")
    tr = pd.read_parquet(PROC/"features_treino.parquet")
    Xr, yr = reamostrar(tr[FEATURES], tr["label"])
    modelo = novo_modelo().fit(Xr, yr)

POS = set(TORNADO.values())
def classificar_dia(dia):
    ft = pd.read_parquet(PROC/f"features_{dia}.parquet")
    el = ft[(ft.n_tx >= MIN_TX) & (ft.rotulo != "router")].copy()
    el["prob"] = modelo.predict_proba(el[FEATURES])[:, 1]
    el["positivo"] = el.rotulo.isin(POS)
    el["dia_ref"] = dia
    infra = ft[(ft.n_tx >= MIN_TX) & (ft.rotulo == "router")].copy()
    if len(infra): infra["prob"] = modelo.predict_proba(infra[FEATURES])[:, 1]
    nao_aval = ft[ft.rotulo.isin(POS) & (ft.n_tx < MIN_TX)][["rotulo", "n_tx"]]
    return el, infra, nao_aval

dias_val = [d for d in DIAS if PAPEIS[d] == "validacao"]
dias_teste = [d for d in DIAS if PAPEIS[d] == "teste"]
cls = {d: classificar_dia(d) for d in DIAS}

# conjuntos de validação: rótulo -> janelas elegíveis com probabilidade
VAL = {d: cls[d][0] for d in dias_val}
if VALIDACAO_2020:
    f20 = pd.read_parquet(PROC/f"features_teste_{VALIDACAO_2020.replace('-','')}.parquet")
    el20 = f20[f20.n_tx >= MIN_TX].copy()
    el20["prob"] = modelo.predict_proba(el20[FEATURES])[:, 1]
    el20["positivo"] = el20["tornado"].notna()
    el20["dia_ref"] = VALIDACAO_2020
    VAL[VALIDACAO_2020] = el20
    print(f"validação {VALIDACAO_2020}: {len(el20):,} janelas elegíveis, {int(el20.positivo.sum())} Tornado (esperado: 3)")
if not VAL:
    raise RuntimeError("Sem conjunto de validação. Defina VALIDACAO_2020 ou use o modo com vários dias de 2026.")

arq_lim = RES/"limiar_congelado_2026.json"
if arq_lim.exists():
    LIMIAR = json.loads(arq_lim.read_text())["limiar"]
    print("limiar já congelado:", LIMIAR)
else:
    els_val = list(VAL.values())
    if sum(int(e.positivo.sum()) for e in els_val) == 0:
        raise RuntimeError("Nenhum Tornado avaliável nos dias de validação: não dá para escolher o limiar. Veja a triagem.")
    LIMIAR, tab_grade = escolher_limiar(els_val)
    tab_grade.to_csv(RES/"limiar_grade_validacao_2026.csv", index=False)
    arq_lim.write_text(json.dumps({"limiar": LIMIAR, "dias_validacao": list(VAL),
                                   "congelado_em": datetime.datetime.now().isoformat(timespec="seconds")}))
    registrar(f"| 13 | Limiar escolhido na validação 2026 | {LIMIAR} | Máximo F1 nos dias {list(VAL)}; congelado antes do teste | {datetime.date.today()} |")
    print("limiar escolhido e CONGELADO:", LIMIAR)
print("\nGrade na validação:")
pd.read_csv(RES/"limiar_grade_validacao_2026.csv")

modelo carregado de modelo_rf.joblib
validação 2020-10-29: 4,038 janelas elegíveis, 3 Tornado (esperado: 3)
limiar escolhido e CONGELADO: 0.85

Grade na validação:


,limiar,sinalizados,tornado,falsos_positivos,fn,precisao,recall,f1
0,0.50,9,3,6,0,0.333,1.0,0.500
1,0.55,9,3,6,0,0.333,1.0,0.500
2,0.60,8,3,5,0,0.375,1.0,0.545
3,0.65,8,3,5,0,0.375,1.0,0.545
4,0.70,8,3,5,0,0.375,1.0,0.545
5,0.75,8,3,5,0,0.375,1.0,0.545
6,0.80,7,3,4,0,0.429,1.0,0.600
7,0.85,7,3,4,0,0.429,1.0,0.600
8,0.90,3,0,3,3,0.000,0.0,0.000
9,0.95,2,0,2,3,0.000,0.0,0.000


In [16]:
print("Positivos avaliáveis na validação:", sum(int(e.positivo.sum()) for e in VAL.values()))
for d, e in VAL.items():
    print(d, metricas_ranking(e))
    if d in cls and len(cls[d][2]): print("   Tornado não avaliável (< MIN_TX):", cls[d][2].values.tolist())
pd.DataFrame([resumo_limiar(list(VAL.values()), l) for l in [0.5, 0.8, LIMIAR]],
             index=["referência 0,5", "referência 0,8", "congelado"])

Positivos avaliáveis na validação: 3
2020-10-29 {'janelas': 4038, 'positivos_avaliaveis': 3, 'ranks_tornado': [6, 6, 7], 'precisao_em_k': 0.0, 'precisao_em_10': 0.3, 'ap': 0.365}


,limiar,sinalizados,tornado,falsos_positivos,fn,precisao,recall,f1
"referência 0,5",0.50,9,3,6,0,0.333,1.0,0.5
"referência 0,8",0.80,7,3,4,0,0.429,1.0,0.6
congelado,0.85,7,3,4,0,0.429,1.0,0.6


---
## Etapa 7 — Teste (rodar **uma vez**)
Aplica o limiar congelado aos dias de teste. Se o teste já tiver sido executado, a célula **avisa e não roda de novo**: repetir após ajustar qualquer coisa invalida o protocolo. (Para ver de novo os números já gravados, abra os CSVs em `results/`.)

In [18]:
marca = RES/"teste_2026_executado.txt"
FORCAR = False     # só mude para True sabendo que o resultado deixa de ser confirmatório
if marca.exists() and not FORCAR:
    print("O TESTE JÁ FOI EXECUTADO em", marca.read_text(), "- resultados em results/. Não rode de novo para ajustar nada.")
else:
    els_teste = [cls[d][0] for d in dias_teste]
    linhas = []
    for d in dias_teste:
        el, infra, nao_aval = cls[d]
        m = metricas_ranking(el); r = resumo_limiar([el], LIMIAR)
        linhas.append({"dia": d, **m, "sinalizados": r["sinalizados"], "tornado_sinalizados": r["tornado"],
                       "falsos_positivos": r["falsos_positivos"], "precisao": r["precisao"], "recall": r["recall"],
                       "tornado_nao_avaliaveis": len(nao_aval)})
    por_dia = pd.DataFrame(linhas); por_dia.to_csv(RES/"teste_2026_por_dia.csv", index=False)

    comp = pd.DataFrame([resumo_limiar(els_teste, l) for l in [LIMIAR, 0.5, 0.8]],
                        index=["congelado (validação)", "referência 0,5", "referência 0,8"])
    comp.to_csv(RES/"teste_2026_resumo.csv")

    pool = pd.concat(els_teste, ignore_index=True)
    rk = {"ap_agrupado": round(float(average_precision_score(pool.positivo, pool.prob)), 3) if pool.positivo.any() else np.nan,
          "positivos_avaliaveis": int(pool.positivo.sum())}
    json.dump(rk, open(RES/"teste_2026_ranking_agrupado.json", "w"))

    sin = pool[pool.prob > LIMIAR].sort_values("prob", ascending=False)
    sin = sin.assign(etherscan="https://etherscan.io/address/" + sin.recipient, categoria="", rotulo_etherscan="", observacao="")
    cols = ["dia_ref", "recipient", "etherscan", "rotulo", "prob", "n_tx"] + FEATURES + ["categoria", "rotulo_etherscan", "observacao"]
    sin[cols].to_csv(RES/"sinalizados_teste_2026.csv", index=False)
    infra_all = pd.concat([cls[d][1].assign(dia_ref=d) for d in dias_teste if len(cls[d][1])], ignore_index=True) if any(len(cls[d][1]) for d in dias_teste) else pd.DataFrame()
    if len(infra_all): infra_all.to_csv(RES/"teste_2026_router.csv", index=False)

    marca.write_text(datetime.datetime.now().isoformat(timespec="seconds"))
    registrar(f"| 14 | Teste 2026 executado | Dias {dias_teste}, limiar {LIMIAR} | Uma única execução com o protocolo congelado | {datetime.date.today()} |")
    print("Resultado por dia de teste:"); display(por_dia)
    print("Agrupado (todos os dias de teste):"); display(comp)
    print("Ranking agrupado:", rk)
    print("\nSinalizados (precisam de categorização no Etherscan):", len(sin), "→ results/sinalizados_teste_2026.csv")
    if len(infra_all): print("Router do Tornado nos dias de teste (fora da contagem):"); display(infra_all[["dia_ref","recipient","n_tx","prob"]])

O TESTE JÁ FOI EXECUTADO em 2026-11-14T22:50:56 - resultados em results/. Não rode de novo para ajustar nada.


---
## O que mandar de volta
1. `results/triagem_dias_2026.csv` e `results/protocolo_2026.json`
2. `results/limiar_grade_validacao_2026.csv` (grade da validação)
3. `results/teste_2026_por_dia.csv`, `results/teste_2026_resumo.csv` e `results/teste_2026_ranking_agrupado.json`
4. `results/sinalizados_teste_2026.csv` (para categorizar os sinalizados no Etherscan)

**Como ler:** compare o resultado do teste com o da validação. Se o teste for pior, a diferença mostra quanto o ajuste do limiar estava inflando o número. A precisão é pessimista porque os "falsos positivos" não foram verificados.